In [1]:
source("/home/user/data2/lit/bin/lit_utils.R")
source("/home/user/data3/lit/project/sORFs/sORFs.utils.R")
lib_text()
lib_plot()

In [2]:
# 设置工作路径
BASE_PATH <- "/home/user/data3/lit/project/sORFs/02-Mass-spec-20250723/analysis/20251028_new_db_search_res"
orfs_file <- file.path(BASE_PATH, "results/orfs_merged_final.tsv")
as_file <- file.path(BASE_PATH, "processed/variant_trace/_as_source.tsv")
snp_file <- file.path(BASE_PATH, "processed/variant_trace/snp_annotated.tsv")

# 输出目录
output_dir <- "../../figures/variant_source"
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

cat("📖 Reading data files...\n")

# ====================================================================
# 1. 读取数据
# ====================================================================

# 读取ORF主表
orfs <- fread_c(orfs_file)
orfs %>% filter(!grepl("Cont",ORF_id)) -> orfs
# orfs %>% filter(ORF_length<=150) -> orfs
nrow(orfs)
# 读取AS来源信息
as_source <- fread_c(as_file)

# 读取SNP注释信息
snp_annot <- fread_c(snp_file)

cat(sprintf("✓ ORFs: %d rows\n", nrow(orfs)))
cat(sprintf("✓ AS source: %d rows\n", nrow(as_source)))
cat(sprintf("✓ SNP annotated: %d rows\n", nrow(snp_annot)))

📖 Reading data files...


[1] 23866

✓ ORFs: 23866 rows
✓ AS source: 23866 rows
✓ SNP annotated: 3894 rows


# AS

In [3]:


# ====================================================================
# 2. 数据整合和分类
# ====================================================================

cat("\n🔗 Merging data...\n")

# 合并AS信息
orfs_with_as <- orfs %>%
  left_join(as_source %>% select(orf_id, structural_category, AS_source),
            by = c("ORF_id" = "orf_id"))

# 定义AS来源分类
orfs_with_as <- orfs_with_as %>%
  mutate(
    AS_category = case_when(
      # Annotated: 完全匹配参考注释
      AS_source == "non_extension"  ~ "Annotated",
      # AS: 新的剪切形式，按structural_category细分
      AS_source == "AS"  ~ "AS",
    TRUE ~ AS_source
    )
  )


# ====================================================================
# 图1: AS来源分类堆叠柱状图
# ====================================================================

cat("\n📊 Creating Figure 1: AS source stacked barplot...\n")

# 统计数据
as_stats <- orfs_with_as %>%
  count(Is_canonical, AS_category) %>%
  group_by(Is_canonical) %>%
  mutate(
    percentage = n / sum(n) * 100,
    label = sprintf("%d\n(%.1f%%)", n, percentage)
  ) %>%
  ungroup()

# 设置AS_category的顺序（从annotated到最novel）
# as_order <- c("Annotated", "Extension", 
#               "AS: Incomplete splice", "AS: Alternative splicing",
#               "AS: Novel in catalog", "AS: Novel not in catalog", 
#               "AS: Other", "Other")

# as_stats$AS_category <- factor(as_stats$AS_category, levels = as_order)

# 绘制堆叠柱状图
p1 <- ggplot(as_stats, aes(x = Is_canonical, y = n, fill = AS_category)) +
  geom_bar(stat = "identity", position = "stack", color = "white", size = 0.3) +
  geom_text(aes(label = n), 
            position = position_stack(vjust = 0.5), 
            size = 3, color = "white", fontface = "bold") +
  scale_fill_brewer(palette = "Set3") +
  scale_x_discrete(labels = c("FALSE" = "Non-canonical\nORFs", 
                              "TRUE" = "Canonical\nORFs")) +
  labs(
    title = "Distribution of Alternative Splicing Sources",
    subtitle = "Comparison between canonical and non-canonical ORFs",
    x = "ORF type",
    y = "Number of ORFs"
  ) +
  theme_3() 

# 保存图1
# ggsave(file.path(output_dir, "fig1_as_source_barplot.pdf"), 
#        plot = p1, width = 10, height = 7)
# ggsave(file.path(output_dir, "fig1_as_source_barplot.png"), 
#        plot = p1, width = 10, height = 7, dpi = 300)

cat("✓ Figure 1 saved\n")

# 打印统计摘要
cat("\n📈 AS Source Statistics:\n")
print(as_stats %>% 
        select(Is_canonical, AS_category, n, percentage) %>%
        arrange(Is_canonical, desc(n)))


🔗 Merging data...

📊 Creating Figure 1: AS source stacked barplot...


Warning message:
“Using `size` aesthetic for lines was deprecated in ggplot2 3.4.0.
ℹ Please use `linewidth` instead.”


✓ Figure 1 saved

📈 AS Source Statistics:
# A tibble: 6 × 4
  Is_canonical AS_category        n percentage
  <lgl>        <chr>          <int>      <dbl>
1 FALSE        Annotated       9814     67.6  
2 FALSE        AS              4081     28.1  
3 FALSE        5UTR_extension   508      3.50 
4 FALSE        3UTR_extension   118      0.813
5 TRUE         Annotated       8761     93.8  
6 TRUE         AS               584      6.25 


In [4]:
tmp <- orfs_with_as %>%
  mutate(
    AS_category = case_when(
      # Annotated: 完全匹配参考注释
      AS_source == "non_extension"  ~ "Annotated",
      # AS: 新的剪切形式，按structural_category细分
      AS_source == "AS"  ~ structural_category,
    TRUE ~ AS_source
    )
  )
table(tmp$Is_canonical,tmp$AS_category)

       
        3UTR_extension 5UTR_extension Annotated antisense fusion genic
  FALSE            118            508      9814        53    154    63
  TRUE               0              0      8761         0      0     3
       
        incomplete-splice_match intergenic moreJunctions novel_in_catalog
  FALSE                     228         78             2             1747
  TRUE                       19          0             0              399
       
        novel_not_in_catalog
  FALSE                 1756
  TRUE                   163

# SNP

In [5]:
snp_annot %>% mutate(mutation_type_collapse=case_when(
mutation_type=="stop_loss" ~ "missense",
mutation_type=="start_gained" ~ "missense",
    TRUE ~ mutation_type
)) -> snp_annot
table(snp_annot$mutation_type_collapse)


  missense synonymous 
      1680       2214 

In [6]:
# ====================================================================
# 2. 计算每个ORF的突变密度
# ====================================================================

cat("\n🔢 Calculating mutation density...\n")

# 统计每个ORF每种突变类型的数量
mutation_counts <- snp_annot %>%
  group_by(orf_id, mutation_type) %>%
  summarise(count = n(), .groups = "drop") %>%
  pivot_wider(names_from = mutation_type, 
              values_from = count, 
              values_fill = 0)
# head(mutation_counts)
# 计算总突变数
mutation_totals <- snp_annot %>%
  group_by(orf_id) %>%
  summarise(
    total_mutations = n(),
    .groups = "drop"
  )
# head(mutation_totals)
# 合并到主表
orfs_with_mutations <- orfs %>%
  left_join(mutation_totals, by = c("ORF_id" = "orf_id")) %>%
  left_join(mutation_counts, by = c("ORF_id" = "orf_id")) 
orfs_with_mutations[is.na(orfs_with_mutations)] <- 0
orfs_with_mutations %>% mutate(missense_density=missense/ORF_length * 100,
                              synonymous_density=synonymous/ORF_length * 100,
                              total_mutations_density=total_mutations/ORF_length * 100)->orfs_with_mutations
# head(orfs_with_mutations)
# 添加分组标签
orfs_with_mutations <- orfs_with_mutations %>%
  mutate(
    # 分组标签
    ORF_group = ifelse(Is_canonical, "Canonical", "Non-canonical")
  )

cat("✓ Mutation density calculated\n")


🔢 Calculating mutation density...
✓ Mutation density calculated


In [7]:
colnames(orfs_with_mutations)

[1] "ORF_id"                            "ORF_type"                         
 [3] "Start_codon"                       "Isoform_id"                       
 [5] "Chr"                               "Strand"                           
 [7] "ORF_seq"                           "ORF_length"                       
 [9] "Geneid"                            "Isoform_structural_category"      
[11] "Isoform_subcategory"               "Is_uniprot"                       
[13] "Is_canonical"                      "Run_occurrence"                   
[15] "RPF_reads"                         "Psites_number"                    
[17] "RPF_RPKM"                          "Psites_RPKM"                      
[19] "RPF_codon_coverage"                "Psites_codon_coverage"            
[21] "FL"                                "FL_TPM"                           
[23] "N"                                 "C"                                
[25] "A"                                 "Gene_type"                        
[27] "Unique_peptide_n"                  "Unique_peptide_n_msfragger_closed"
[29] "mean_relative_iBAQ"                "total_mutations"                  
[31] "synonymous"                        "missense"                         
[33] "stop_loss"                         "start_gained"                     
[35] "missense_density"                  "synonymous_density"               
[37] "total_mutations_density"           "ORF_group"

In [8]:
# ====================================================================
# 3. 统计检验
# ====================================================================

cat("\n📊 Statistical testing...\n")

# 准备长格式数据用于统计检验
mutation_long <- orfs_with_mutations %>%
  select(ORF_id, ORF_group, ORF_length,
         missense_density, synonymous_density, total_mutations_density) %>%
  pivot_longer(cols = ends_with("_density"),
               names_to = "mutation_type",
               values_to = "density") %>%
  mutate(
    mutation_type = str_remove(mutation_type, "_density"),
    mutation_type = str_to_title(mutation_type)
  )

# Wilcoxon检验（针对每种突变类型）
stat_results <- mutation_long %>%
  group_by(mutation_type) %>%
  summarise(
    canonical_median = median(density[ORF_group == "Canonical"]),
    canonical_mean = mean(density[ORF_group == "Canonical"]),
    non_canonical_median = median(density[ORF_group == "Non-canonical"]),
    non_canonical_mean = mean(density[ORF_group == "Non-canonical"]),
    p_value = wilcox.test(
      density[ORF_group == "Canonical"],
      density[ORF_group == "Non-canonical"]
    )$p.value,
    .groups = "drop"
  ) %>%
  mutate(
    p_adjusted = p.adjust(p_value, method = "BH"),
    significance = case_when(
      p_adjusted < 0.001 ~ "***",
      p_adjusted < 0.01 ~ "**",
      p_adjusted < 0.05 ~ "*",
      TRUE ~ "ns"
    )
  )

cat("\n📈 Statistical Test Results:\n")
print(stat_results, width = Inf)


📊 Statistical testing...

📈 Statistical Test Results:
# A tibble: 3 × 8
  mutation_type   canonical_median canonical_mean non_canonical_median
  <chr>                      <dbl>          <dbl>                <dbl>
1 Missense                       0         0.0232                    0
2 Synonymous                     0         0.0367                    0
3 Total_mutations                0         0.0606                    0
  non_canonical_mean   p_value p_adjusted significance
               <dbl>     <dbl>      <dbl> <chr>       
1             0.0660 5.44e- 64  5.44e- 64 ***         
2             0.0287 1.35e-258  4.05e-258 ***         
3             0.101  6.60e-236  9.90e-236 ***         


In [9]:
head(mutation_long)
mutation_long -> df
nrow(df)/3

ORF_id,ORF_group,ORF_length,mutation_type,density
<chr>,<chr>,<int>,<chr>,<dbl>
PB.10027.4:chr12:-|10|1315:112:1117|canonical|ATG,Canonical,334,Missense,0.0000000
PB.10027.4:chr12:-|10|1315:112:1117|canonical|ATG,Canonical,334,Synonymous,0.2994012
PB.10027.4:chr12:-|10|1315:112:1117|canonical|ATG,Canonical,334,Total_mutations,0.2994012
PB.1003.3:chr1:+|10|581:117:393|canonical|ATG,Canonical,91,Missense,0.0000000
PB.1003.3:chr1:+|10|581:117:393|canonical|ATG,Canonical,91,Synonymous,0.0000000
PB.1003.3:chr1:+|10|581:117:393|canonical|ATG,Canonical,91,Total_mutations,0.0000000


[1] 23866

In [10]:
# library(dplyr)
# library(ggplot2)

# # df: 至少包含 ORF_group, mutation_type, density 三列
# keep_types <- c("Missense","Synonymous","Total_mutations")

# df %>%
#   filter(mutation_type %in% keep_types) %>%
#   mutate(
#     mutation_type = factor(mutation_type, levels = keep_types),
#     ORF_group = factor(ORF_group)            # 如需固定顺序可在此处设 levels
#   ) %>%
#   ggplot(aes(x = ORF_group, y = density, fill = ORF_group)) +
#   geom_violin(trim = FALSE, alpha = 0.6) +
#   geom_boxplot(width = 0.15, outlier.shape = NA, color = "black") +
#   facet_wrap(~ mutation_type, nrow = 1, scales = "free_y") +  # 三个类型并排
#   labs(x = "ORF group", y = "Density") +
#   theme_3()


In [11]:
snp_file_1 <- file.path(BASE_PATH, "processed/variant_trace/manual_results.tsv")
fread_c(snp_file_1) -> snps_1
head(snps_1)
head(orfs)

,orf_id,mutation_position,mutation_type,aa_change,ref_aa,alt_aa,orf_found,has_peptides,is_covered,covering_peptides,covering_samples,peptide_positions,total_peptides_for_orf
,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<lgl>,<lgl>,<lgl>,<chr>,<chr>,<chr>,<int>
1,PB.22.22:chr1:-|23|1332:342:834|noncoding|ATG,100,missense,P100L,P,L,TRUE,TRUE,FALSE,,,,1
2,PB.24.14:chr1:-|6|2774:36:2286|canonical|ATG,615,synonymous,L615L,L,L,TRUE,TRUE,FALSE,,,,7
3,PB.24.14:chr1:-|6|2774:36:2286|canonical|ATG,394,synonymous,T394T,T,T,TRUE,TRUE,FALSE,,,,7
4,PB.24.14:chr1:-|6|2774:36:2286|canonical|ATG,306,synonymous,E306E,E,E,TRUE,TRUE,FALSE,,,,7
5,PB.24.14:chr1:-|6|2774:36:2286|canonical|ATG,300,missense,I300V,I,V,TRUE,TRUE,FALSE,,,,7
6,PB.25.4:chr1:+|2|2547:96:2025|canonical|ATG,203,synonymous,A203A,A,A,TRUE,TRUE,FALSE,,,,3


,ORF_id,ORF_type,Start_codon,Isoform_id,Chr,Strand,ORF_seq,ORF_length,Geneid,Isoform_structural_category,⋯,Psites_codon_coverage,FL,FL_TPM,N,C,A,Gene_type,Unique_peptide_n,Unique_peptide_n_msfragger_closed,mean_relative_iBAQ
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>,<int>,<dbl>
1,PB.10027.4:chr12:-|10|1315:112:1117|canonical|ATG,canonical,ATG,PB.10027.4,chr12,-,MATLKEKLIAPVAEEEATVPNNKITVVGVGQVGMACAISILGKSLADELALVDVLEDKLKGEMMDLQHGSLFLQTPKIVADKDYSVTANSKIVVVTAGVRQQEGESRLNLVQRNVNVFKFIIPQIVKYSPDCIIIVVSNPVDILTYVTWKLSGLPKHRVIGSGCNLDSARFRYLMAEKLGIHPSSCHGWILGEHGDSSVAVWSGVNVAGVSLQELNPEMGTDNDSENWKEVHKMVVESAYEVIKLKGYTNWAIGLSVADLIESMLKNLSRIHPVSTMVKGMYGIENEVFLSLPCILNARGLTSVINQKLKDDEVAQLKKSADTLWDIQKDLKDL,334,LDHB,full-splice_match,⋯,315.50600,4163,568.983154,0.3698762,0.07221995,0.2210481,protein_coding,52,35,1.158578e-03
2,PB.1003.3:chr1:+|10|581:117:393|canonical|ATG,canonical,ATG,PB.1003.3,chr1,+,MGLEDEQKMLTESGDPEEEEEEEEELVDPLTTVREQCEQLEKCVKARERLELCDERVSSRSHTEEDCTEELFDFLHARDHCVAHKLFNNLK,91,UQCRH,full-splice_match,⋯,43.97800,284,38.816051,1.6756973,82.17642184,41.9260596,protein_coding,11,9,6.918883e-04
3,PB.10043.1:chr12:+|7|1792:131:1436|canonical|ATG,canonical,ATG,PB.10043.1,chr12,+,MDSVEKGAATSVSNPRGRPSRGRPPKLQRNSRGGQGRGVEKPPHLAALILARGGSKGIPLKNIKHLAGVPLIGWVLRAALDSGAFQSVWVSTDHDEIENVAKQFGAQVHRRSSEVSKDSSTSLDAIIEFLNYHNEVDIVGNIQATSPCLHPTDLQKVAEMIREEGYDSVFSVVRRHQFRWSEIQKGVREVTEPLNLNPAKRPRRQDWDGELYENGSFYFAKRHLIEMGYLQGGKMAYYEMRAEHSVDIDVDIDWPIAEQRVLRYGYFGKEKLKEIKLLVCNIDGCLTNGHIYVSGDQKEIISYDVKDAIGISLLKKSGIEVRLISERACSKQTLSSLKLDCKMEVSVSDKLAVVDEWRKEMGLCWKEVAYLGNEVSDEECLKRVGLSGAPADACSTAQKAVGYICKCNGGRGAIREFAEHICLLMEKVNNSCQK,434,CMAS,full-splice_match,⋯,31.85480,2273,310.665070,1.9916688,16.97763009,9.4846494,protein_coding,22,21,6.400495e-05
4,PB.1009.1:chr1:-|10|2772:187:838|canonical|ATG,canonical,ATG,PB.1009.1,chr1,-,MALCLKQVFAKDKTFRPRKRFEPGTQRFELYKKAQASLKSGLDLRSVVRLPPGENIDDWIAVHVVDFFNRINLIYGTMAERCSETSCPVMAGGPRYEYRWQDERQYRRPAKLSAPRYMALLMDWIEGLINDEEVFPTRVGVPFPKNFQQVCTKILTRLFRVFVHVYIHHFDSILSMGAEAHVNTCYKHFYYFIREFSLVDQRELEPLREMTERICH,216,MOB3C,full-splice_match,⋯,1.41204,22,3.006877,0.3449610,0.35701687,0.3509890,protein_coding,1,0,NA
5,PB.10167.18:chr12:+|20|2518:201:1527|canonical|ATG,canonical,ATG,PB.10167.18,chr12,+,MDDDDFGGFEAAETFDGGSGETQTTSPAIPWAAFPAVSGVHLSPSSPEIVLDRDHSSSIGCLSSDAIISSPENTHAANSIVSQTIPKAQIQQSTHTHLDISLFPLGLTDEKSNGTIALVDDSEDPGANVSNIQLQQKISSLEIKLKVSEEEKQRIKQDVESLMEKHNVLEKGFLKEKEQEAISFQDRYKELQEKHKQELEDMRKAGHEALSIIVDEYKALLQSSVKQQVEAIEKQYISAIEKQAHKCEELLNAQHQRLLEMLDTEKELLKEKIKEALIQQSQEQKEILEKCLEEERQRNKEALVSAAKLEKEAVKDAVLKVVEEERKNLEKAHAEERELWKTEHAKDQEKVSQEIQKAIQEQRKISQETVKAAIIEEQKRSEKAVEEAVKRTRDELIEYIKEQKRLDQVIRQRSLSSLELFLSCAQKQLSALIATEPVDIE,441,CCDC91,full-splice_match,⋯,20.53060,55,7.517193,0.7278217,2.94518451,1.8365031,protein_coding,32,29,1.488966e-05
6,PB.10216.11:chr12:+|30|2937:422:2675|noncoding|ATG,noncoding,ATG,PB.10216.11,chr12,+,MEEIKPASASCVSKEKPSKVSDLISRFEGGSSLSNYSDLKKESAVNLNAPRTPGRHGLTTTPQQKLLSQHLPQRQGNDTDKTQGAQTCVANGVMAAQNQMECEEEKAATLSSDTSIQASEPLLDTHIVNGERDETATAPASPTTDSCDGNASDSSYRTPGIGPVLPLEERGAETETKVQERENGESPLELEQLDQHHEMKETNEQKLHKIANELLLTERAYVNRLDLLDQVFYCKLLEEANRGSFPAEMVNKIFSNISSINAFHSKFLLPELEKRMQEWETTPRIGDILQKLAPFLKMYGEYVKGFDNAMELVKNMTERIPQFKSVVEEIQKQKICGSLTLQHHMLEPVQRIPRYEMLLKDYLRKLPPDSLDWNDAKKSLEIISTAASHSNSAIRKMENLKKLLEIYEMLGEEEDIVNPSNELIKEGQILKLAARNTSAQERYLFLFNNMLLYCVPKFSLVGSKFTVRTRVGIDGMKIVETQNEEYPHTFQVSGKERTLELQASSAQDKEEWIKALQETIDAFHQRHETFRNAIAKDNDIHSEVSTAELGKRAPRWIRDNEVTMCMKCKEPFNALTRRRHHCRACGYVVCWKCSDYKAQLEYDGGKLSKVCKDCYQIISGFTDSEEKKRKGILEIESAEVSGNSVVCSFLQYMEKSKPWQKAWCVIPKQDPLVLYMYGAPQDVRAQATIPLLGYVVDEMPRSADLPHSFKLTQSKSVHSFAADSEELKQKWLKVILLAVTERAERNKIRN,750,FGD4,novel_in_catalog,⋯,15.88400,3,0.410029,4.3771021,15.64060387,10.0088530,protein_coding,61,51,3.111357e-05


In [12]:
merge(snps_1,orfs,by.x="orf_id",by.y="ORF_id") -> snp_2
snp_2 %>% mutate(mutation_type_collapse=case_when(
mutation_type=="stop_loss" ~ "missense",
mutation_type=="start_gained" ~ "missense",
    TRUE ~ mutation_type
)) -> snp_3
head(snp_3)

,orf_id,mutation_position,mutation_type,aa_change,ref_aa,alt_aa,orf_found,has_peptides,is_covered,covering_peptides,⋯,FL,FL_TPM,N,C,A,Gene_type,Unique_peptide_n,Unique_peptide_n_msfragger_closed,mean_relative_iBAQ,mutation_type_collapse
,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<lgl>,<lgl>,<lgl>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>,<int>,<dbl>,<chr>
1,PB.10027.4:chr12:-|10|1315:112:1117|canonical|ATG,87,synonymous,T87T,T,T,TRUE,TRUE,FALSE,,⋯,4163,568.983154,0.3698762,0.07221995,0.2210481,protein_coding,52,35,1.158578e-03,synonymous
2,PB.1004.25:chr1:+|3|2053:52:1792|canonical|ATG,299,synonymous,C299C,C,C,TRUE,TRUE,FALSE,,⋯,98,13.394271,0.7119992,0.21871670,0.4653580,protein_coding,28,26,2.706921e-05,synonymous
3,PB.1004.9:chr1:+|3|1522:24:1179|canonical|ATG,51,missense,T51A,T,A,TRUE,TRUE,FALSE,,⋯,16,2.186820,0.6371582,0.74246579,0.6898120,protein_coding,1,1,NA,missense
4,PB.10048.74:chr12:+|12|1594:186:780|noncoding|ATG,33,synonymous,E33E,E,E,TRUE,TRUE,FALSE,,⋯,2,0.273352,3.1038694,2.12224948,2.6130594,protein_coding,2,2,7.814561e-05,synonymous
5,PB.10097.3:chr12:+|33|2253:480:1980|canonical|ATG,141,missense,L141V,L,V,TRUE,TRUE,FALSE,,⋯,3,0.410029,0.2889804,0.44653722,0.3677588,protein_coding,1,1,1.148361e-05,missense
6,PB.10097.3:chr12:+|33|2253:480:1980|canonical|ATG,197,missense,C197S,C,S,TRUE,TRUE,FALSE,,⋯,3,0.410029,0.2889804,0.44653722,0.3677588,protein_coding,1,1,1.148361e-05,missense


In [13]:
count(snp_3,Is_canonical,mutation_type_collapse)
filter(snp_3,mutation_type_collapse=="missense") %>% filter(is_covered=="TRUE") %>% count(.,Is_canonical,mutation_type_collapse)

Is_canonical,mutation_type_collapse,n
<lgl>,<chr>,<int>
FALSE,missense,602
FALSE,synonymous,432
TRUE,missense,1078
TRUE,synonymous,1782


Is_canonical,mutation_type_collapse,n
<lgl>,<chr>,<int>
FALSE,missense,102
TRUE,missense,111


In [14]:
table(orfs_with_mutations$ORF_group)


    Canonical Non-canonical 
         9345         14521 